[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 中等难度：从零实现 LayerNorm

**在 LayerNorm 或 RMSNorm 中，需要计算激活值的方差来进行归一化，确保模型训练的稳定性。**

请**从头实现** Layer Normalization（层归一化）。

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

其中 $\mu$ 和 $\sigma^2$ 是沿着**最后一个维度**计算得到的。

### 函数签名
```python
def my_layer_norm(
    x: torch.Tensor,      # 输入张量
    gamma: torch.Tensor,   # 缩放参数（大小与最后一个维度相同）
    beta: torch.Tensor,    # 偏移参数（大小与最后一个维度相同）
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### 要求
- **禁止**使用 `F.layer_norm` 或 `torch.nn.LayerNorm`
- 只对**最后一个维度**进行归一化
- 必须支持 autograd（自动求导）

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

/Users/xiaohui/Documents/project/TorchCode/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor.var()计算某个维度的方差，使用有偏方差估计。

unbiased=False (有偏 vs 无偏) ★ 核心重点这是统计学中一个经典的区别：unbiased=True (默认)：计算的是样本方差。分母使用的是 $n - 1$（贝塞尔修正）。$$s^2 = \frac{\sum (x_i - \mu)^2}{n - 1}$$unbiased=False：计算的是总体方差。分母使用的是 $n$。$$\sigma^2 = \frac{\sum (x_i - \mu)^2}{n}$$为什么 LayerNorm 要用 unbiased=False？在深度学习的归一化层（LayerNorm, BatchNorm, InstanceNorm）的标准定义中，通常直接使用当前观测数据的均值和方差，分母固定为 $n$。为了与 torch.nn.LayerNorm 的官方行为保持 100% 一致，必须手动设置 unbiased=False。

In [ ]:
# ✏️ 在这里写你的实现

def my_layer_norm(x, gamma, beta, eps=1e-5):
    # 1. 计算均值 (mu)
    # dim=-1 表示沿着最后一个维度计算
    # keepdim=True 保证结果 shape 为 (..., 1)，方便后续广播计算
    mean = x.mean(dim=-1, keepdim=True) # 最后一维度进行归一化
    
    # 2. 计算方差 (sigma^2)
    # var = E[(x - mean)^2]
    var = x.var(dim=-1, keepdim=True, unbiased=False) # Variance 有偏方差估计，将样本固定为 n
    
    # 3. 归一化 (Standardization)
    # x_hat = (x - mu) / sqrt(var + eps)
    x_normalized = (x - mean) / torch.sqrt(var + eps)
    
    # 4. 线性变换 (Scale and Shift)
    # 使用训练好的 gamma 和 beta 进行重构
    out = gamma * x_normalized + beta # 缩放和偏差都和最后一个维度相同
    
    return out

In [ ]:
dtype = torch.get_default_dtype()
x = torch.tensor([[10,20,30,40], [50,60,70,80]]).to(torch.float32)

mean = x.mean(dim=-1, keepdim=True) # 维度在直观上的感觉顺序是 批，列，行

print(mean)

tensor([[25.],
        [65.]])


In [ ]:
# 🧪 调试用
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("你的输出均值：", out.mean(dim=-1))   # 应该接近 0
print("你的输出标准差：", out.std(dim=-1))    # 应该接近 1
print("与官方实现一致？", torch.allclose(out, ref, atol=1e-4))

In [ ]:
# ✅ 提交验证
from torch_judge import check
check("layernorm")